In [ ]:
import cv2 
import nibabel as nib
import numpy as np
import os 
import tifffile

base_path = '/Users/calvinyong/Desktop/UW Madison/Research Materials/Dataset/ArabidopsisDataset/Finetuning/PreCorrected'
seg_dir = os.path.join(base_path, "Segmentation", "Ensemble")

images = [img for img in os.listdir(base_path) if img.endswith('.png') or img.endswith('.tif')]
if not images:
    raise FileNotFoundError(f'No .png/.tif images in {base_path}')

# Store the .nii.gz image and segmentation next to the originals
# (image -> <stem>_img.nii.gz, segmentation -> <stem>.nii.gz, same as tif_to_nii.py)
for image in images:
    stem, ext = os.path.splitext(image)

    # convert the image itself
    image_path = os.path.join(base_path, image)
    if ext == '.tif':
        img = tifffile.imread(image_path)
    else:
        img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    img_nii_path = os.path.join(base_path, stem + '_img.nii.gz')
    nib.save(nib.Nifti1Image(img.T, affine=np.eye(4)), img_nii_path)
    print(f'Saved image to {img_nii_path}')

    # convert the segmentation
    seg_path = os.path.join(seg_dir, stem + '.png')
    seg = cv2.imread(seg_path, cv2.IMREAD_UNCHANGED)
    if seg is None:
        print(f'No segmentation found for {image} at {seg_path}, skipping')
        continue
    seg = seg.astype(np.uint8).T

    seg_nii = nib.Nifti1Image(seg, affine=np.eye(4))
    seg_nii_path = os.path.join(base_path, stem + '.nii.gz')
    nib.save(seg_nii, seg_nii_path)
    print(f'Saved segmentation to {seg_nii_path}')

In [ ]:
# Run after editing in ITK-SNAP: convert the edited masks (<stem>.nii.gz) back to .png
import cv2
import nibabel as nib
import numpy as np
import os

base_path = '/Users/calvinyong/Desktop/UW Madison/Research Materials/Dataset/ArabidopsisDataset/Finetuning/Corrected'
# separate folder so the original predictions in Segmentation/Ensemble are not overwritten
out_dir = os.path.join(base_path, "Corrected")
os.makedirs(out_dir, exist_ok=True)

# masks only: skip the image volumes (<stem>_img.nii.gz)
masks = [f for f in os.listdir(base_path) if f.endswith('.nii.gz') and not f.endswith('_img.nii.gz')]
if not masks:
    raise FileNotFoundError(f'No mask .nii.gz files in {base_path}')

for mask in masks:
    stem = mask[:-len('.nii.gz')]
    nii = nib.load(os.path.join(base_path, mask))
    seg = np.asanyarray(nii.dataobj).squeeze().T.astype(np.uint8)  # undo the .T applied on export

    out_path = os.path.join(out_dir, stem + '.png')
    cv2.imwrite(out_path, seg)
    print(f'Saved {out_path}')